# Naive Baseline

Lag-based baseline: predict y[t] = har_ma_125[t] (125-period rolling mean of adjusted RV).

In [ ]:
import os
import subprocess
import sys

# Clone repo if not present
if not os.path.exists("harxhar"):
    subprocess.run(["git", "clone", "https://github.com/your-org/harxhar.git"], check=True)
os.chdir("harxhar/colab")

# Install dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas", "scikit-learn", "pyarrow", "tqdm"]
)

sys.path.insert(0, ".")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import resolve_har_lags, generate_har_features, apply_horizon_shift
from src.evaluation import apply_duan_smearing
from src.transforms import robust_transform

HORIZON = 1
TRAIN_WINDOW_DAYS = 500
DATA_PATH = "all30min"
PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY

# Load + transform + features
df = load_raw_data(DATA_PATH)
adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline
df, feature_names = generate_har_features(df, target_col="adj_RV")

max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

print(f"Data: {X.shape[0]} samples, {len(feature_names)} features")
print(f"Feature names: {feature_names}")

In [ ]:
# Naive model: y_pred[t] = har_ma_125[t]
lag_125_index = feature_names.index("har_ma_125")
print(f"Using feature index {lag_125_index} (har_ma_125) as naive predictor")

oos_start = TRAIN_WINDOW
X_oos = X[oos_start:]
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]
preds = X_oos[:, lag_125_index]

# Autocorrelation of adjusted RV
from pandas.plotting import autocorrelation_plot  # noqa: E402

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
autocorrelation_plot(pd.Series(y_oos[:5000]), ax=axes[0])
axes[0].set_title("Autocorrelation of adj_RV")
axes[0].set_xlim(0, 200)
axes[1].scatter(preds[:2000], y_oos[:2000], alpha=0.3, s=5)
axes[1].set_xlabel("Predicted (har_ma_125)")
axes[1].set_ylabel("Actual (adj_RV)")
axes[1].set_title("Naive Baseline: Predicted vs Actual")
plt.tight_layout()
plt.show()

pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

In [ ]:
from src.evaluation import calculate_metrics

results_df = pd.DataFrame(
    {
        "date": dates_oos,
        "horizon": HORIZON,
        "true_adj": y_oos,
        "pred_adj": preds,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
# export
"""Naive lag-based baseline volatility forecast executor."""

import os

import numpy as np
import pandas as pd

from src.evaluation import apply_duan_smearing
from src.executor import build_executor_parser, load_and_transform
from src.transforms import (
    PERIODS_PER_DAY,
    apply_horizon_shift,
    generate_har_features,
    resolve_har_lags,
)


In [ ]:
# Verify the exported imports resolved: the names main() relies on must be bound.
for _name in [
    "argparse",
    "os",
    "np",
    "pd",
    "load_raw_data",
    "robust_transform",
    "resolve_har_lags",
    "generate_har_features",
    "apply_horizon_shift",
    "PERIODS_PER_DAY",
    "apply_duan_smearing",
]:
    assert _name in dir(), f"missing import: {_name}"
print("all imports bound")

In [ ]:
# export
def main() -> None:
    args = build_executor_parser("Naive baseline backtest").parse_args()

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df, _ = load_and_transform(
        args.data_path,
        exog_cols=[],
        target_use_diurnal=False,
        target_winsor_window=None,
        dropna_with_exog=False,
    )

    df, feature_names = generate_har_features(df, target_col="adj_RV")
    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end
    X_chunk = X[start:end]
    y_chunk = y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})")

    lag_125_index = feature_names.index("har_ma_125")
    oos_start = train_win_periods
    X_oos = X_chunk[oos_start:]
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]
    preds = X_oos[:, lag_125_index]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame(
        {
            "date": dates_oos,
            "horizon": args.horizon,
            "true_adj": y_oos,
            "pred_adj": preds,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    print(f"Saved {len(results)} rows -> {args.output_file}")


In [ ]:
# Smoke-test main(): inspect its argparse surface without executing a full backtest.
import inspect

assert callable(main)
sig = inspect.signature(main)
assert list(sig.parameters) == [], f"main() should take no args, got {list(sig.parameters)}"
print("main() signature OK:", sig)

In [ ]:
# export
if __name__ == "__main__":
    main()

In [ ]:
from _exporter import export_notebook

export_notebook("ml_baseline.ipynb", "../src/ml_baseline.py")